In [1]:
# read csv data file
import csv
f = open("module2_quiz_P1.csv")
csvfile = csv.DictReader(f)

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [2]:
# define set of Segments
Segments = []
for row in table:
    if row['Segment'] != 'Cost per ad':
        Segments.append(row['Segment'])
        
# define set of Outlets
Outlets = headers[:]
Outlets.remove('Segment')
Outlets.remove('Minimum Impressions')

In [3]:
# define parameters
Cost = {}
for o in Outlets:
    Cost[o] = float(table[-1][o])

Demand = {}
for row in table:
    if row['Segment'] != 'Cost per ad':
        Demand[row['Segment']] = float(row['Minimum Impressions'])
        
Impressions = {}
for row in table:
    if row['Segment'] != 'Cost per ad':
        for o in Outlets:
            Impressions[(row['Segment'],o)] = float(row[o])

In [4]:
Segments

['WWC', 'TTL', 'NEN', 'ILC']

In [5]:
Outlets

['R', 'TV1', 'TV2', 'TV3', 'W', 'M']

In [6]:
Cost

{'R': 75.0, 'TV1': 275.0, 'TV2': 300.0, 'TV3': 250.0, 'W': 100.0, 'M': 90.0}

In [7]:
Demand

{'WWC': 50000.0, 'TTL': 70000.0, 'NEN': 25000.0, 'ILC': 52000.0}

In [8]:
Impressions

{('WWC', 'R'): 500.0,
 ('WWC', 'TV1'): 3000.0,
 ('WWC', 'TV2'): 2000.0,
 ('WWC', 'TV3'): 4000.0,
 ('WWC', 'W'): 25.0,
 ('WWC', 'M'): 50.0,
 ('TTL', 'R'): 3.0,
 ('TTL', 'TV1'): 150.0,
 ('TTL', 'TV2'): 250.0,
 ('TTL', 'TV3'): 300.0,
 ('TTL', 'W'): 1700.0,
 ('TTL', 'M'): 2000.0,
 ('NEN', 'R'): 1500.0,
 ('NEN', 'TV1'): 2500.0,
 ('NEN', 'TV2'): 4000.0,
 ('NEN', 'TV3'): 2000.0,
 ('NEN', 'W'): 150.0,
 ('NEN', 'M'): 0.0,
 ('ILC', 'R'): 0.0,
 ('ILC', 'TV1'): 500.0,
 ('ILC', 'TV2'): 1000.0,
 ('ILC', 'TV3'): 1500.0,
 ('ILC', 'W'): 100.0,
 ('ILC', 'M'): 0.0}

In [9]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [10]:
# variables
category = mdl.continuous_var_dict(Outlets, lb=0, name="category")

In [11]:
# objective
objective = mdl.sum(category[i]*Cost[i] for i in Outlets)
mdl.minimize(objective)

In [12]:
# define the minimum impressions constraints:
for s in Segments:
    mdl.add_constraint(mdl.sum(category[i]*Impressions[(s,i)] for i in Outlets) >= Demand[s])

In [13]:
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00180984,status='optimal')

In [14]:
mdl.print_solution()

objective: 11348.667
status: OPTIMAL_SOLUTION(2)
  category_TV3=34.667
  category_M=29.800
